
# LFH-aware multi-prefix stability check — Gemma2-2B-IT / AGNews

This notebook implements a lightweight practical remedy for Local Feature Homeostasis (LFH): **do not trust feature rankings from a single fixed few-shot prefix until they survive final-demonstration variation**.

Experiment design:

1. Use Gemma2-2B-IT and the Gemma Scope residual SAE at the middle layer (`layer 12`).
2. Sample AGNews test queries, label-balanced.
3. For each query, keep demonstrations 1–7 fixed and vary only demonstration 8 across several balanced alternatives.
4. Measure query-side SAE pre/post-activations at token positions controlled by one variable only:
   ```python
   query_analysis_positions = [-2]
   ```
   Here `-1` is the final query-span token, `-2` is the second-final token, etc.
5. Select **label-specific** candidate features. For label `L`, the score is:
   ```text
   mean preactivation on gold-label-L query items
   - mean preactivation on all other query items
   ```
6. For each label, report the top-20 label-specific features from the ordinary single-prefix baseline and check whether they remain stable across final-demo variants.
7. Plot all diagnostics with Plotly.

The notebook does not read or write analysis caches. Re-running the notebook recomputes the activations from scratch.


This updated version also adds a **feature-level causal validation** section. It tests whether features that pass the LFH-aware multi-prefix stability rule are more causally relevant to the gold-label probability than features flagged as LFH-sensitive.


In [1]:

# ============================================================
# 0. Imports and configuration
# ============================================================

import os
import re
import json
import math
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from IPython.display import display


def set_seed(seed: int):
    print("seed:", seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PYTHONHASHSEED"] = str(seed)


# -----------------------------
# User-facing experiment knobs
# -----------------------------
seed = 42
set_seed(seed)

MODEL_NAME = "google/gemma-2-2b-it"
MODEL_SHORT_NAME = "Gemma2-2B-IT"
TARGET_LAYER = 12

# Gemma Scope 2B PT residual SAE, matching the LFH middle-layer setup.
SAE_REPO_ID = "google/gemma-scope-2b-pt-res"
SAE_WIDTH = "width_16k"
SAE_AVERAGE_L0 = "average_l0_82"
SAE_FILENAME = f"layer_{TARGET_LAYER}/{SAE_WIDTH}/{SAE_AVERAGE_L0}/params.npz"

# Multi-prefix stability experiment.
N_QUERY_TOTAL = 100              # label-balanced test queries; use smaller values for debugging
N_SHOT = 8                       # demonstrations per prefix
N_PREFIX_VARIANTS = 8            # with AGNews labels, 8 gives 2 final demos per label
FINAL_DEMO_SLOT = N_SHOT         # vary the last demonstration: Example 8

# Label-specific feature selection / stability settings.
LABEL_SPECIFIC_TOP_N = 20        # top-20 features per label
RETENTION_TOP_K = 20             # retention is computed against the top-K label-specific ranking in each variant
STABLE_MIN_RETENTION = 0.70
STABLE_MIN_BASELINE_RATIO = 0.50

# One token-position variable only.
# -1 is the final query-span token, -2 is the second-final token, etc.
query_analysis_positions = [-2, -1]

# Optional local-control condition: vary an earlier demo while keeping the final demo fixed.
# This approximately doubles the number of forward passes, so it is disabled by default.
RUN_EARLIER_DEMO_CONTROL = False
EARLIER_DEMO_CONTROL_SLOT = 3

# Runtime.
DTYPE = torch.bfloat16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 4

# Optional HF token for higher rate limits / gated checkpoints.
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
HF_KWARGS = {"token": HF_TOKEN} if HF_TOKEN else {}

print("MODEL_NAME:", MODEL_NAME)
print("TARGET_LAYER:", TARGET_LAYER)
print("SAE_REPO_ID:", SAE_REPO_ID)
print("SAE_FILENAME:", SAE_FILENAME)
print("query_analysis_positions:", query_analysis_positions)
print("DEVICE:", DEVICE)


/home/ikhyuncho23/ml-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


seed: 42
MODEL_NAME: google/gemma-2-2b-it
TARGET_LAYER: 12
SAE_REPO_ID: google/gemma-scope-2b-pt-res
SAE_FILENAME: layer_12/width_16k/average_l0_82/params.npz
query_analysis_positions: [-2, -1]
DEVICE: cuda


In [2]:

# ============================================================
# 1. Load AGNews, tokenizer, model, and Gemma Scope SAE
# ============================================================

label_to_words_agnews = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Technology",
}
word_to_label_agnews = {v: k for k, v in label_to_words_agnews.items()}
NUM_LABELS = len(label_to_words_agnews)


def load_agnews_splits():
    """Use the namespaced HF dataset id to avoid recent hf:// URI errors."""
    candidates = ["fancyzhx/ag_news", "ag_news"]
    errors = []
    for dataset_id in candidates:
        try:
            print(f"Loading AGNews from {dataset_id!r}...")
            return load_dataset(dataset_id, split=["train", "test"], **HF_KWARGS)
        except Exception as e:
            errors.append((dataset_id, type(e).__name__, str(e)))
            print(f"Failed to load {dataset_id!r}: {type(e).__name__}: {e}")
    raise RuntimeError("Could not load AGNews. Errors:\n" + "\n\n".join(map(str, errors)))


dataset = load_agnews_splits()
train_data_agnews = dataset[0]
test_data_agnews = dataset[1]

train_indices_by_label = defaultdict(list)
test_indices_by_label = defaultdict(list)
for i, row in enumerate(train_data_agnews):
    train_indices_by_label[int(row["label"])].append(i)
for i, row in enumerate(test_data_agnews):
    test_indices_by_label[int(row["label"])].append(i)

print("Train counts:", {label_to_words_agnews[k]: len(v) for k, v in train_indices_by_label.items()})
print("Test counts:", {label_to_words_agnews[k]: len(v) for k, v in test_indices_by_label.items()})


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True,
    **HF_KWARGS,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Left padding makes causal labels natural and matches the previous notebooks.
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
    **HF_KWARGS,
)
model.eval()


def get_model_input_device(model):
    return model.get_input_embeddings().weight.device


def get_layer_device(model, layer_idx: int):
    block = model.get_submodule(f"model.layers.{layer_idx}")
    try:
        return next(block.parameters()).device
    except StopIteration:
        return get_model_input_device(model)


MODEL_INPUT_DEVICE = get_model_input_device(model)
TARGET_LAYER_DEVICE = get_layer_device(model, TARGET_LAYER)
print("MODEL_INPUT_DEVICE:", MODEL_INPUT_DEVICE)
print("TARGET_LAYER_DEVICE:", TARGET_LAYER_DEVICE)


class GemmaScopeJumpReLUSAE(torch.nn.Module):
    """Minimal inference wrapper for Gemma Scope JumpReLU residual-stream SAEs."""

    def __init__(self, params, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        def get_param(name, *aliases):
            for key in (name,) + aliases:
                if key in params:
                    value = params[key]
                    if isinstance(value, np.ndarray):
                        value = torch.from_numpy(value)
                    return value.detach() if torch.is_tensor(value) else torch.tensor(value)
            raise KeyError(f"Missing SAE parameter {name!r}. Available keys: {sorted(params.keys())}")

        W_enc = get_param("W_enc")
        W_dec = get_param("W_dec")
        b_enc = get_param("b_enc")
        b_dec = get_param("b_dec")
        threshold = get_param("threshold", "thresholds")

        if W_enc.ndim != 2 or W_dec.ndim != 2:
            raise ValueError(f"Expected 2D W_enc/W_dec, got {tuple(W_enc.shape)} and {tuple(W_dec.shape)}")

        d_model = int(W_enc.shape[0])
        n_features = int(W_enc.shape[1])

        if W_dec.shape == (d_model, n_features):
            W_dec = W_dec.T.contiguous()
        if W_dec.shape != (n_features, d_model):
            raise ValueError(f"Expected W_dec shape {(n_features, d_model)}, got {tuple(W_dec.shape)}")
        if b_enc.numel() != n_features:
            raise ValueError(f"Expected b_enc length {n_features}, got {b_enc.numel()}")
        if b_dec.numel() != d_model:
            raise ValueError(f"Expected b_dec length {d_model}, got {b_dec.numel()}")
        if threshold.numel() != n_features:
            raise ValueError(f"Expected threshold length {n_features}, got {threshold.numel()}")

        self.register_buffer("W_enc", W_enc.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))

        self.k = None
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        # Gemma Scope folds any preprocessing into the encoder parameters for these checkpoints.
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, torch.relu(pre), torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


def load_gemma_scope_jumprelu_sae(repo_id: str, filename: str, device=DEVICE, dtype=DTYPE):
    print("Downloading/loading Gemma Scope SAE checkpoint:", repo_id, filename)
    path = hf_hub_download(repo_id=repo_id, filename=filename, **HF_KWARGS)
    with np.load(path, allow_pickle=False) as params_np:
        params = {k: torch.from_numpy(params_np[k]) for k in params_np.files}
    sae = GemmaScopeJumpReLUSAE(params, device=device, dtype=dtype)
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"mean_threshold={float(sae.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae


sae = load_gemma_scope_jumprelu_sae(
    repo_id=SAE_REPO_ID,
    filename=SAE_FILENAME,
    device=DEVICE,
    dtype=DTYPE,
)


Loading AGNews from 'fancyzhx/ag_news'...
Train counts: {'Business': 30000, 'Technology': 30000, 'Sports': 30000, 'World': 30000}
Test counts: {'Business': 1900, 'Technology': 1900, 'Sports': 1900, 'World': 1900}


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████████████| 288/288 [00:00<00:00, 791.81it/s]


MODEL_INPUT_DEVICE: cuda:0
TARGET_LAYER_DEVICE: cuda:0
Downloading/loading Gemma Scope SAE checkpoint: google/gemma-scope-2b-pt-res layer_12/width_16k/average_l0_82/params.npz
Loaded SAE: d_model=2304, n_features=16384, mean_threshold=3.1322


In [3]:

# ============================================================
# 2. Prompt sampling and rendering
# ============================================================

rng = random.Random(seed)

INSTRUCTION = """You are given news articles and their topics.
Valid topics: World, Sports, Business, Technology.
For each example, identify the correct topic."""


def balanced_label_sequence(n: int, num_labels: int = NUM_LABELS, rng: random.Random | None = None):
    """Return labels as evenly as possible, shuffled."""
    rng = rng or random
    base = n // num_labels
    rem = n % num_labels
    labels = []
    for lab in range(num_labels):
        labels.extend([lab] * base)
    extra = list(range(num_labels))
    rng.shuffle(extra)
    labels.extend(extra[:rem])
    rng.shuffle(labels)
    return labels


def sample_train_indices_for_labels(label_seq, rng: random.Random, exclude=None):
    exclude = set(exclude or [])
    chosen = []
    used = set(exclude)
    for lab in label_seq:
        pool = [idx for idx in train_indices_by_label[int(lab)] if idx not in used]
        if not pool:
            raise RuntimeError(f"No available training examples for label {lab} after exclusions.")
        idx = rng.choice(pool)
        chosen.append(idx)
        used.add(idx)
    return chosen


def sample_label_balanced_test_indices(n_total: int, rng: random.Random):
    label_seq = balanced_label_sequence(n_total, num_labels=NUM_LABELS, rng=rng)
    out = []
    used = set()
    for lab in label_seq:
        pool = [idx for idx in test_indices_by_label[int(lab)] if idx not in used]
        idx = rng.choice(pool)
        out.append(idx)
        used.add(idx)
    return out


def format_demo(demo_number: int, row) -> str:
    return (
        f"Example {demo_number}:\n"
        f"News article:\n{row['text']}\n"
        f"Topic:\n{label_to_words_agnews[int(row['label'])]}"
    )


def render_single_user_prompt(demo_rows, query_row):
    """
    Render all demonstrations and the test query inside one Gemma user turn.

    Returns:
      prompt: full model input string
      query_char_span: [start, end) character span of the query block inside prompt
    """
    if len(demo_rows) != N_SHOT:
        raise ValueError(f"Expected {N_SHOT} demos, got {len(demo_rows)}")

    demo_text = "\n\n".join(format_demo(i + 1, row) for i, row in enumerate(demo_rows))
    query_block = f"Test article:\n{query_row['text']}\nTopic:\n"
    user_content = f"{INSTRUCTION}\n\n{demo_text}\n\n{query_block}"
    prompt = f"<bos><start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n"

    query_start = prompt.index(query_block)
    query_end = query_start + len(query_block)

    # Sanity checks: one user turn, one model-generation header, all demos in the user turn.
    if prompt.count("<start_of_turn>user") != 1:
        raise AssertionError("Prompt should contain exactly one user turn.")
    if prompt.count("<start_of_turn>model") != 1:
        raise AssertionError("Prompt should contain exactly one model-generation header.")
    if prompt.count("<end_of_turn>") != 1:
        raise AssertionError("Prompt should contain exactly one user-turn terminator.")
    if len(re.findall(r"(?m)^Example\s+\d+:", prompt)) != N_SHOT:
        raise AssertionError(f"Prompt should contain exactly {N_SHOT} numbered demo headers.")

    return prompt, (query_start, query_end)


def build_final_demo_variant_prompts():
    """
    Build Q x V prompts.

    For each fixed query:
      - demos 1..7 are fixed;
      - demo 8 varies across N_PREFIX_VARIANTS alternatives;
      - final-demo labels are as balanced as possible.
    """
    test_indices = sample_label_balanced_test_indices(N_QUERY_TOTAL, rng)
    prompts = []
    metadata = []

    for q_local_idx, test_idx in enumerate(test_indices):
        query_row = test_data_agnews[int(test_idx)]

        fixed_label_seq = balanced_label_sequence(N_SHOT - 1, rng=rng)
        fixed_demo_indices = sample_train_indices_for_labels(fixed_label_seq, rng=rng)

        final_label_seq = balanced_label_sequence(N_PREFIX_VARIANTS, rng=rng)
        final_demo_indices = sample_train_indices_for_labels(
            final_label_seq,
            rng=rng,
            exclude=fixed_demo_indices,
        )

        fixed_demo_rows = [train_data_agnews[int(i)] for i in fixed_demo_indices]

        for variant_idx, final_demo_idx in enumerate(final_demo_indices):
            demo_indices = list(fixed_demo_indices) + [int(final_demo_idx)]
            demo_rows = fixed_demo_rows + [train_data_agnews[int(final_demo_idx)]]
            prompt, query_char_span = render_single_user_prompt(demo_rows, query_row)
            prompts.append(prompt)
            metadata.append({
                "query_local_idx": int(q_local_idx),
                "variant_idx": int(variant_idx),
                "query_test_idx": int(test_idx),
                "query_label": int(query_row["label"]),
                "query_label_word": label_to_words_agnews[int(query_row["label"])],
                "fixed_demo_indices": [int(x) for x in fixed_demo_indices],
                "fixed_demo_labels": [int(train_data_agnews[int(x)]["label"]) for x in fixed_demo_indices],
                "final_demo_idx": int(final_demo_idx),
                "final_demo_label": int(train_data_agnews[int(final_demo_idx)]["label"]),
                "final_demo_label_word": label_to_words_agnews[int(train_data_agnews[int(final_demo_idx)]["label"])],
                "query_char_start": int(query_char_span[0]),
                "query_char_end": int(query_char_span[1]),
            })

    return prompts, metadata


prompts, prompt_metadata = build_final_demo_variant_prompts()
print(f"Built {len(prompts)} prompts = {N_QUERY_TOTAL} queries x {N_PREFIX_VARIANTS} final-demo variants.")
print("First prompt excerpt:\n")
print(prompts[0][:2000])

meta_df = pd.DataFrame(prompt_metadata)
print("\nQuery label counts:")
display(meta_df.drop_duplicates("query_local_idx")["query_label_word"].value_counts().sort_index().to_frame("count"))
print("Final-demo label counts across all variants:")
display(meta_df["final_demo_label_word"].value_counts().sort_index().to_frame("count"))


Built 800 prompts = 100 queries x 8 final-demo variants.
First prompt excerpt:

<bos><start_of_turn>user
You are given news articles and their topics.
Valid topics: World, Sports, Business, Technology.
For each example, identify the correct topic.

Example 1:
News article:
Hamm #39;s legacy should be for Olympic ideal Even though he has fled the perceived hostile atmosphere of Athens and the Olympic village for more friendly terrain in the United States, there is still time for Paul Hamm to do what #39;s right.
Topic:
Sports

Example 2:
News article:
Greenspan rejects talk of throwing dollar a lifeline ALAN GREENSPAN, Chairman of the US Federal Reserve, undercut lingering hopes of staging an operation to support the dollar at a meeting of bankers in Frankfurt yesterday 
Topic:
Business

Example 3:
News article:
Google Desktop Search app lets you 'google' your Windows PC Google Inc. unveiled a desktop search application that lets users catalog and search through e-mail, files, Web histo

,count
query_label_word,
Business,25
Sports,25
Technology,25
World,25


Final-demo label counts across all variants:


,count
final_demo_label_word,
Business,200
Sports,200
Technology,200
World,200


In [4]:

# ============================================================
# 3. Token-span helpers and model/SAE extraction
# ============================================================


def token_positions_for_char_span(prompt: str, char_start: int, char_end: int):
    """Map a prompt character span to unpadded token indices."""
    enc = tokenizer(
        prompt,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    offsets = enc["offset_mapping"]
    positions = []
    for tok_idx, (s, e) in enumerate(offsets):
        # Skip empty special offsets if any appear.
        if e <= s:
            continue
        # Token overlaps the query span.
        if max(int(s), int(char_start)) < min(int(e), int(char_end)):
            positions.append(tok_idx)
    if not positions:
        raise RuntimeError("No token positions found for query character span.")
    return positions


def select_query_positions(all_query_positions):
    """
    Select query-span token positions using query_analysis_positions only.

    Offset semantics:
      -1 -> final query-span token
      -2 -> second-final query-span token
      -3 -> third-final query-span token
    """
    selected = []
    n = len(all_query_positions)
    for off in query_analysis_positions:
        off = int(off)
        if off >= 0:
            raise ValueError("query_analysis_positions must contain negative offsets, e.g. [-1, -2].")
        idx = n + off
        if idx < 0 or idx >= n:
            raise IndexError(f"Offset {off} is out of range for query span with {n} tokens.")
        selected.append(all_query_positions[idx])
    return selected


# Precompute selected unpadded token positions for every prompt.
for meta, prompt in zip(prompt_metadata, prompts):
    all_positions = token_positions_for_char_span(
        prompt=prompt,
        char_start=meta["query_char_start"],
        char_end=meta["query_char_end"],
    )
    meta["all_query_token_positions"] = [int(x) for x in all_positions]
    meta["selected_query_token_positions"] = [int(x) for x in select_query_positions(all_positions)]
    meta["num_query_tokens"] = int(len(all_positions))
    meta["num_selected_query_tokens"] = int(len(meta["selected_query_token_positions"]))

print("Selected query-token counts:")
display(pd.Series([m["num_selected_query_tokens"] for m in prompt_metadata]).describe().to_frame("selected_query_tokens"))
print("Example selected offsets:", query_analysis_positions)
print("First prompt selected unpadded token positions:", prompt_metadata[0]["selected_query_token_positions"])


def clear_all_hooks(model: torch.nn.Module):
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()


@torch.no_grad()
def get_layer_output(model, input_ids, attention_mask, layer_idx: int):
    """Run a forward pass and capture model.layers[layer_idx] output, shape [B, S, D]."""
    captured = {}

    def hook_fn(module, inputs, output):
        captured["x"] = output[0].detach() if isinstance(output, tuple) else output.detach()
        return output

    block = model.get_submodule(f"model.layers.{layer_idx}")
    handle = block.register_forward_hook(hook_fn)
    try:
        _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    finally:
        handle.remove()

    if "x" not in captured:
        raise RuntimeError(f"Failed to capture output from layer {layer_idx}")
    return captured["x"]


@torch.no_grad()
def collect_query_feature_stats_for_prompts(prompts, metadata):
    """
    Return per-prompt query-side feature statistics:
      pre_by_prompt:  [P, F] mean SAE pre-activation over selected query tokens
      post_by_prompt: [P, F] mean SAE post-activation over selected query tokens
      freq_by_prompt: [P, F] fraction of selected query tokens with post-activation > 0
    """

    n_prompts = len(prompts)
    n_features = int(sae.n_features)
    pre_by_prompt = np.zeros((n_prompts, n_features), dtype=np.float32)
    post_by_prompt = np.zeros((n_prompts, n_features), dtype=np.float32)
    freq_by_prompt = np.zeros((n_prompts, n_features), dtype=np.float32)

    model_device = get_model_input_device(model)
    model.eval()
    sae.eval()

    for start in range(0, n_prompts, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n_prompts)
        batch_prompts = prompts[start:end]
        batch_meta = metadata[start:end]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            return_tensors="pt",
            add_special_tokens=False,
        )
        input_ids = enc["input_ids"].to(model_device)
        attention_mask = enc["attention_mask"].to(model_device)

        hidden = get_layer_output(model, input_ids, attention_mask, TARGET_LAYER)  # [B, S, D]
        batch_seq_len = int(input_ids.shape[1])

        for b, meta in enumerate(batch_meta):
            unpadded_len = int(attention_mask[b].sum().item())
            pad_shift = batch_seq_len - unpadded_len
            selected_positions = [pad_shift + int(pos) for pos in meta["selected_query_token_positions"]]
            selected_positions = [pos for pos in selected_positions if 0 <= pos < batch_seq_len]
            if not selected_positions:
                raise RuntimeError(f"No valid selected token positions for prompt index {start + b}.")

            x = hidden[b, selected_positions, :]
            pre = sae.encode_pre(x).float()
            post = sae.activation_fn(pre.to(dtype=sae.dtype)).float()

            pre_by_prompt[start + b] = pre.mean(dim=0).detach().cpu().numpy()
            post_by_prompt[start + b] = post.mean(dim=0).detach().cpu().numpy()
            freq_by_prompt[start + b] = (post > 0).float().mean(dim=0).detach().cpu().numpy()

        del hidden, input_ids, attention_mask, enc

        print(f"Processed prompts {end}/{n_prompts}")

    return pre_by_prompt, post_by_prompt, freq_by_prompt, metadata


Selected query-token counts:


,selected_query_tokens
count,800.0
mean,2.0
std,0.0
min,2.0
25%,2.0
50%,2.0
75%,2.0
max,2.0


Example selected offsets: [-2, -1]
First prompt selected unpadded token positions: [577, 578]


In [5]:

# ============================================================
# 4. Run final-demo-variant extraction
# ============================================================

pre_by_prompt, post_by_prompt, freq_by_prompt, prompt_metadata_loaded = collect_query_feature_stats_for_prompts(
    prompts=prompts,
    metadata=prompt_metadata,
)

# Reshape from [Q*V, F] to [Q, V, F]. Prompts were built query-major, then variant-major.
n_features = pre_by_prompt.shape[-1]
pre_by_query_variant = pre_by_prompt.reshape(N_QUERY_TOTAL, N_PREFIX_VARIANTS, n_features)
post_by_query_variant = post_by_prompt.reshape(N_QUERY_TOTAL, N_PREFIX_VARIANTS, n_features)
freq_by_query_variant = freq_by_prompt.reshape(N_QUERY_TOTAL, N_PREFIX_VARIANTS, n_features)

print("pre_by_query_variant:", pre_by_query_variant.shape)
print("post_by_query_variant:", post_by_query_variant.shape)
print("freq_by_query_variant:", freq_by_query_variant.shape)


Processed prompts 4/800
Processed prompts 8/800
Processed prompts 12/800
Processed prompts 16/800
Processed prompts 20/800
Processed prompts 24/800
Processed prompts 28/800
Processed prompts 32/800
Processed prompts 36/800
Processed prompts 40/800
Processed prompts 44/800
Processed prompts 48/800
Processed prompts 52/800
Processed prompts 56/800
Processed prompts 60/800
Processed prompts 64/800
Processed prompts 68/800
Processed prompts 72/800
Processed prompts 76/800
Processed prompts 80/800
Processed prompts 84/800
Processed prompts 88/800
Processed prompts 92/800
Processed prompts 96/800
Processed prompts 100/800
Processed prompts 104/800
Processed prompts 108/800
Processed prompts 112/800
Processed prompts 116/800
Processed prompts 120/800
Processed prompts 124/800
Processed prompts 128/800
Processed prompts 132/800
Processed prompts 136/800
Processed prompts 140/800
Processed prompts 144/800
Processed prompts 148/800
Processed prompts 152/800
Processed prompts 156/800
Processed pr

In [6]:

# ============================================================
# 5. Label-specific feature selection and multi-prefix stability
# ============================================================

# Query labels in the same query-major order used by pre_by_query_variant/post_by_query_variant.
query_meta_df = meta_df.drop_duplicates("query_local_idx").sort_values("query_local_idx").reset_index(drop=True)
query_labels = query_meta_df["query_label"].to_numpy(dtype=np.int64)
if len(query_labels) != N_QUERY_TOTAL:
    raise AssertionError(f"Expected {N_QUERY_TOTAL} query labels, got {len(query_labels)}")

print("Query label counts used for label-specific scoring:")
display(pd.Series(query_labels).map(label_to_words_agnews).value_counts().sort_index().to_frame("count"))


def label_vs_other_means(arr_qvf: np.ndarray, labels: np.ndarray):
    """
    Compute label-vs-other means for arrays shaped [Q, V, F].

    Returns:
      label_mean: [L, V, F]
      other_mean: [L, V, F]
    """
    if arr_qvf.shape[0] != labels.shape[0]:
        raise ValueError(f"arr_qvf has {arr_qvf.shape[0]} queries but labels has {labels.shape[0]}")
    label_mean = np.zeros((NUM_LABELS, arr_qvf.shape[1], arr_qvf.shape[2]), dtype=np.float32)
    other_mean = np.zeros_like(label_mean)
    for lab in range(NUM_LABELS):
        mask = labels == lab
        other = ~mask
        if mask.sum() == 0 or other.sum() == 0:
            raise RuntimeError(f"Need at least one label-{lab} and one non-label-{lab} query.")
        label_mean[lab] = arr_qvf[mask].mean(axis=0)
        other_mean[lab] = arr_qvf[other].mean(axis=0)
    return label_mean, other_mean


# Label-specific score requested by the user:
#   average preactivation on label L - average preactivation on all other labels.
label_mean_pre, other_mean_pre = label_vs_other_means(pre_by_query_variant, query_labels)
label_mean_post, other_mean_post = label_vs_other_means(post_by_query_variant, query_labels)
label_mean_freq, other_mean_freq = label_vs_other_means(freq_by_query_variant, query_labels)
label_specific_pre_gap = label_mean_pre - other_mean_pre  # [label, variant, feature]

# Rank every feature by label-specific preactivation gap within each label and prefix variant.
label_variant_ranks = np.empty_like(label_specific_pre_gap, dtype=np.int32)  # [L, V, F]
for lab in range(NUM_LABELS):
    for v in range(N_PREFIX_VARIANTS):
        order = np.argsort(-label_specific_pre_gap[lab, v])
        ranks_v = np.empty(n_features, dtype=np.int32)
        ranks_v[order] = np.arange(1, n_features + 1, dtype=np.int32)
        label_variant_ranks[lab, v] = ranks_v


def summarize_label_specific_feature(lab: int, feat: int, rank: int):
    scores_v = label_specific_pre_gap[lab, :, feat]
    ranks_v = label_variant_ranks[lab, :, feat]
    baseline = float(scores_v[0])
    mean_score = float(scores_v.mean())
    std_score = float(scores_v.std(ddof=0))
    min_score = float(scores_v.min())
    max_score = float(scores_v.max())
    score_range = float(max_score - min_score)
    eps = 1e-12

    if baseline > eps:
        min_over_baseline = float(min_score / baseline)
        mean_over_baseline = float(mean_score / baseline)
    else:
        min_over_baseline = np.nan
        mean_over_baseline = np.nan

    retention_rate = float((ranks_v <= RETENTION_TOP_K).mean())
    stable = bool(
        (retention_rate >= STABLE_MIN_RETENTION)
        and np.isfinite(min_over_baseline)
        and (min_over_baseline >= STABLE_MIN_BASELINE_RATIO)
    )

    return {
        "label": int(lab),
        "label_word": label_to_words_agnews[int(lab)],
        "label_specific_rank": int(rank),
        "feature_idx": int(feat),
        "baseline_label_specific_pre_gap": baseline,
        "mean_label_specific_pre_gap_across_prefixes": mean_score,
        "std_label_specific_pre_gap_across_prefixes": std_score,
        "min_label_specific_pre_gap_across_prefixes": min_score,
        "max_label_specific_pre_gap_across_prefixes": max_score,
        "range_label_specific_pre_gap_across_prefixes": score_range,
        "mean_over_baseline": mean_over_baseline,
        "min_over_baseline": min_over_baseline,
        f"retention_rate_top_{RETENTION_TOP_K}": retention_rate,
        "worst_variant_rank": int(ranks_v.max()),
        "best_variant_rank": int(ranks_v.min()),
        "stable_by_rule": stable,
        "baseline_mean_pre_label": float(label_mean_pre[lab, 0, feat]),
        "baseline_mean_pre_other": float(other_mean_pre[lab, 0, feat]),
        "baseline_mean_post_label": float(label_mean_post[lab, 0, feat]),
        "baseline_mean_post_other": float(other_mean_post[lab, 0, feat]),
        "baseline_activation_frequency_label": float(label_mean_freq[lab, 0, feat]),
        "baseline_activation_frequency_other": float(other_mean_freq[lab, 0, feat]),
        "mean_pre_label_across_prefixes": float(label_mean_pre[lab, :, feat].mean()),
        "mean_pre_other_across_prefixes": float(other_mean_pre[lab, :, feat].mean()),
        "mean_post_label_across_prefixes": float(label_mean_post[lab, :, feat].mean()),
        "mean_post_other_across_prefixes": float(other_mean_post[lab, :, feat].mean()),
        "activation_frequency_label_across_prefixes": float(label_mean_freq[lab, :, feat].mean()),
        "activation_frequency_other_across_prefixes": float(other_mean_freq[lab, :, feat].mean()),
    }


label_top_features = {}
rows = []
for lab in range(NUM_LABELS):
    baseline_scores = label_specific_pre_gap[lab, 0]  # ordinary single-prefix setting, variant 0
    order = np.argsort(-baseline_scores)
    top_feats = [int(x) for x in order[:LABEL_SPECIFIC_TOP_N]]
    label_top_features[lab] = top_feats
    for rank, feat in enumerate(top_feats, start=1):
        rows.append(summarize_label_specific_feature(lab=lab, feat=feat, rank=rank))

label_specific_stability_df = pd.DataFrame(rows)
label_specific_stability_df = label_specific_stability_df.sort_values(
    by=["label", "label_specific_rank"],
    ascending=[True, True],
).reset_index(drop=True)

lfh_sensitive_label_specific_df = label_specific_stability_df[~label_specific_stability_df["stable_by_rule"]].sort_values(
    by=["label", "label_specific_rank"],
    ascending=[True, True],
).reset_index(drop=True)

print(f"Top-{LABEL_SPECIFIC_TOP_N} label-specific features per label, selected by baseline preactivation gap.")
display(label_specific_stability_df)

print("\nOriginally top-ranked label-specific features flagged as LFH-sensitive by the simple multi-prefix rule:")
display(lfh_sensitive_label_specific_df)

print("\nPer-label tables:")
for lab in range(NUM_LABELS):
    label_word = label_to_words_agnews[lab]
    print("=" * 100)
    print(f"{label_word}: top-{LABEL_SPECIFIC_TOP_N} label-specific features")
    display(label_specific_stability_df[label_specific_stability_df["label"] == lab].reset_index(drop=True))

# Union of all top label-specific features, useful for optional controls.
candidate_label_feature_pairs = [
    (int(row["label"]), int(row["feature_idx"]))
    for _, row in label_specific_stability_df.iterrows()
]
candidate_feature_ids = sorted({feat for _, feat in candidate_label_feature_pairs})
print(f"Union of selected features across labels: {len(candidate_feature_ids)} unique features.")


Query label counts used for label-specific scoring:


,count
Business,25
Sports,25
Technology,25
World,25


Top-20 label-specific features per label, selected by baseline preactivation gap.


,label,label_word,label_specific_rank,feature_idx,baseline_label_specific_pre_gap,mean_label_specific_pre_gap_across_prefixes,std_label_specific_pre_gap_across_prefixes,min_label_specific_pre_gap_across_prefixes,max_label_specific_pre_gap_across_prefixes,range_label_specific_pre_gap_across_prefixes,...,baseline_mean_post_label,baseline_mean_post_other,baseline_activation_frequency_label,baseline_activation_frequency_other,mean_pre_label_across_prefixes,mean_pre_other_across_prefixes,mean_post_label_across_prefixes,mean_post_other_across_prefixes,activation_frequency_label_across_prefixes,activation_frequency_other_across_prefixes
0,0,World,1,14090,6.002006,6.293857,0.246875,5.998874,6.648398,0.649525,...,4.387813,0.566146,0.46,0.100000,4.615151,-1.678705,4.530000,0.543203,0.4475,0.088333
1,0,World,2,1041,4.656667,3.807292,0.962563,2.403334,5.158334,2.755000,...,18.629999,14.466666,0.90,0.826667,16.968750,13.161459,16.945000,13.461250,0.8600,0.818333
2,0,World,3,5185,3.586107,3.751735,0.208816,3.520117,4.075338,0.555221,...,2.885313,0.757187,0.50,0.140000,3.228184,-0.523551,2.778867,0.600169,0.4250,0.116667
3,0,World,4,14599,3.075547,3.144004,0.239948,2.881536,3.503099,0.621562,...,7.624375,4.496459,0.94,0.680000,7.876778,4.732774,7.693125,4.678463,0.9200,0.702500
4,0,World,5,2600,2.712956,2.593507,0.171621,2.273672,2.777604,0.503932,...,0.880625,0.053021,0.20,0.013333,0.455767,-2.137741,0.839375,0.026784,0.1875,0.007500
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,3,Technology,16,11807,1.578333,1.223698,0.199412,0.925417,1.578333,0.652916,...,0.000000,0.000000,0.00,0.000000,-12.651406,-13.875105,0.000000,0.000000,0.0000,0.000000
76,3,Technology,17,11970,1.436165,1.398313,0.087326,1.187803,1.470817,0.283014,...,0.237813,0.024062,0.06,0.006667,0.581709,-0.816604,0.222461,0.015247,0.0600,0.004167
77,3,Technology,18,9059,1.354323,0.822209,0.322120,0.385169,1.354323,0.969154,...,1.145625,0.663646,0.18,0.100000,0.291084,-0.531125,0.897305,0.867865,0.1525,0.128333
78,3,Technology,19,4626,1.353281,0.963695,0.183517,0.677611,1.353281,0.675671,...,0.139062,0.052292,0.04,0.013333,0.034861,-0.928834,0.063750,0.029349,0.0175,0.007500



Originally top-ranked label-specific features flagged as LFH-sensitive by the simple multi-prefix rule:


...

Business: top-20 label-specific features


,label,label_word,label_specific_rank,feature_idx,baseline_label_specific_pre_gap,mean_label_specific_pre_gap_across_prefixes,std_label_specific_pre_gap_across_prefixes,min_label_specific_pre_gap_across_prefixes,max_label_specific_pre_gap_across_prefixes,range_label_specific_pre_gap_across_prefixes,...,baseline_mean_post_label,baseline_mean_post_other,baseline_activation_frequency_label,baseline_activation_frequency_other,mean_pre_label_across_prefixes,mean_pre_other_across_prefixes,mean_post_label_across_prefixes,mean_post_other_across_prefixes,activation_frequency_label_across_prefixes,activation_frequency_other_across_prefixes
0,2,Business,1,13946,5.054987,5.457988,0.282014,5.036758,5.838171,0.801413,...,4.600625,1.220521,0.52,0.153333,4.933453,-0.524536,4.762344,1.213932,0.5375,0.157500
1,2,Business,2,4300,3.724766,3.787959,0.266098,3.446615,4.168450,0.721836,...,2.595625,0.231146,0.34,0.046667,2.390010,-1.397949,2.662656,0.271003,0.3300,0.053333
2,2,Business,3,2992,3.072539,2.861650,0.252510,2.496719,3.276771,0.780052,...,2.111875,1.186667,0.38,0.193333,2.628325,-0.233325,2.096133,1.086484,0.3775,0.180833
3,2,Business,4,1041,2.910001,3.345625,0.761662,1.931666,4.345000,2.413334,...,17.350000,14.893333,0.94,0.813333,16.622501,13.276875,16.593750,13.578333,0.9125,0.800833
4,2,Business,5,9059,2.634844,2.663862,0.131482,2.464935,2.857877,0.392943,...,1.913750,0.407604,0.28,0.066667,1.672324,-0.991538,2.175000,0.441966,0.2875,0.083333
5,2,Business,6,15109,2.502174,2.388188,0.119962,2.195742,2.504492,0.308750,...,0.890938,0.068333,0.18,0.013333,1.584712,-0.803477,0.788750,0.057708,0.1825,0.010833
6,2,Business,7,14526,2.397789,2.402397,0.149419,2.126596,2.650741,0.524145,...,1.808125,0.868333,0.26,0.160000,2.010875,-0.391522,1.763555,0.781081,0.2800,0.130000
7,2,Business,8,10466,2.209063,2.115091,0.130593,1.844102,2.236914,0.392812,...,0.961250,0.191250,0.18,0.040000,1.233135,-0.881956,0.979922,0.177344,0.1900,0.040833
8,2,Business,9,3635,1.920579,2.015106,0.169285,1.663483,2.289701,0.626217,...,0.000000,0.000000,0.00,0.000000,-1.770754,-3.785860,0.066563,0.000000,0.0175,0.000000
9,2,Business,10,2009,1.759791,1.874551,0.293710,1.185469,2.214478,1.029010,...,10.231875,8.987291,0.72,0.633333,10.402422,8.527870,10.421719,8.852969,0.7500,0.641667


Technology: top-20 label-specific features


,label,label_word,label_specific_rank,feature_idx,baseline_label_specific_pre_gap,mean_label_specific_pre_gap_across_prefixes,std_label_specific_pre_gap_across_prefixes,min_label_specific_pre_gap_across_prefixes,max_label_specific_pre_gap_across_prefixes,range_label_specific_pre_gap_across_prefixes,...,baseline_mean_post_label,baseline_mean_post_other,baseline_activation_frequency_label,baseline_activation_frequency_other,mean_pre_label_across_prefixes,mean_pre_other_across_prefixes,mean_post_label_across_prefixes,mean_post_other_across_prefixes,activation_frequency_label_across_prefixes,activation_frequency_other_across_prefixes
0,3,Technology,1,2245,5.990352,5.189663,0.417398,4.581204,5.990352,1.409147,...,4.856875,0.595312,0.60,0.093333,5.114721,-0.074941,4.461758,0.697904,0.5450,0.120833
1,3,Technology,2,6810,4.333333,3.129792,0.744131,2.070000,4.333333,2.263333,...,0.905000,0.215000,0.12,0.033333,-11.083750,-14.213541,0.420000,0.366458,0.0650,0.043333
2,3,Technology,3,14906,3.828646,4.364593,0.381433,3.828646,5.182304,1.353659,...,0.090000,0.047917,0.02,0.006667,-2.229680,-6.594273,0.305547,0.050104,0.0600,0.006667
3,3,Technology,4,2992,3.224831,3.022920,0.216378,2.636081,3.275091,0.639010,...,2.354375,1.105833,0.38,0.193333,2.749277,-0.273643,2.226836,1.042917,0.3800,0.180000
4,3,Technology,5,14559,3.063504,2.819923,0.153678,2.553506,3.063504,0.509998,...,1.605937,0.081458,0.26,0.013333,1.913003,-0.906920,1.567617,0.081120,0.2575,0.012500
5,3,Technology,6,6049,2.937812,2.854588,0.098932,2.660299,2.964310,0.304011,...,0.541875,0.000000,0.10,0.000000,0.481665,-2.372923,0.722852,0.000000,0.1600,0.000000
6,3,Technology,7,12609,2.892683,2.797567,0.129139,2.567787,2.961745,0.393958,...,0.100000,0.000000,0.02,0.000000,-2.572114,-5.369681,0.089648,0.000000,0.0225,0.000000
7,3,Technology,8,9257,2.661328,2.446862,0.207724,2.130312,2.717422,0.587110,...,4.983125,2.520208,0.78,0.500000,4.964063,2.517200,4.793906,2.585794,0.7450,0.502500
8,3,Technology,9,826,2.577344,2.119546,0.282130,1.735195,2.577344,0.842149,...,1.510313,0.306042,0.28,0.060000,1.193638,-0.925908,1.279609,0.369622,0.2175,0.070833
9,3,Technology,10,9706,2.494525,2.027484,0.267191,1.743346,2.494525,0.751179,...,3.028125,0.966979,0.46,0.166667,3.627334,1.599849,2.736562,1.034805,0.4100,0.174167


Union of selected features across labels: 68 unique features.


In [7]:

# ============================================================
# 6. Plotly diagnostics and heatmaps
# ============================================================

summary = {
    "model": MODEL_SHORT_NAME,
    "target_layer": TARGET_LAYER,
    "n_queries": N_QUERY_TOTAL,
    "n_prefix_variants": N_PREFIX_VARIANTS,
    "label_specific_top_n_per_label": LABEL_SPECIFIC_TOP_N,
    "retention_top_k": RETENTION_TOP_K,
    "stable_min_retention": STABLE_MIN_RETENTION,
    "stable_min_baseline_ratio": STABLE_MIN_BASELINE_RATIO,
    "query_analysis_positions": query_analysis_positions,
    "stable_count": int(label_specific_stability_df["stable_by_rule"].sum()),
    "sensitive_count": int((~label_specific_stability_df["stable_by_rule"]).sum()),
}
print(json.dumps(summary, indent=2))

# Per-variant final-demo label distribution. This checks whether variant index itself is label-confounded.
variant_label_counts = pd.crosstab(meta_df["variant_idx"], meta_df["final_demo_label_word"])
print("Final-demo label counts by variant index:")
display(variant_label_counts)

fig_counts = px.imshow(
    variant_label_counts,
    text_auto=True,
    aspect="auto",
    title="Final-demo label counts by prefix-variant index",
    labels=dict(x="Final-demo label", y="Variant index", color="count"),
)
fig_counts.show()


def plot_label_specific_heatmap(lab: int, normalize_to_baseline: bool = False):
    """Plot top label-specific features for one label across final-demo variants."""
    feature_ids = label_top_features[int(lab)]
    mat = label_specific_pre_gap[int(lab), :, feature_ids].T  # [features, variants]
    if normalize_to_baseline:
        denom = np.abs(mat[:, [0]]) + 1e-12
        mat_to_plot = mat / denom
        colorbar_title = "pre-gap / |baseline gap|"
        title_suffix = "ratio to baseline"
    else:
        mat_to_plot = mat
        colorbar_title = "label-specific preactivation gap"
        title_suffix = "raw gap"

    y_labels = [f"L{TARGET_LAYER}-{int(f)}" for f in feature_ids]
    x_labels = [f"v{v}" for v in range(N_PREFIX_VARIANTS)]
    fig = go.Figure(
        data=go.Heatmap(
            z=mat_to_plot,
            x=x_labels,
            y=y_labels,
            colorscale="RdBu",
            reversescale=True,
            zmid=0.0,
            colorbar=dict(title=colorbar_title),
            hovertemplate=(
                "feature=%{y}<br>variant=%{x}<br>"
                + colorbar_title
                + "=%{z:.4f}<extra></extra>"
            ),
        )
    )
    fig.update_layout(
        title=f"{label_to_words_agnews[int(lab)]}: top-{LABEL_SPECIFIC_TOP_N} label-specific features across final-demo variants ({title_suffix})",
        xaxis_title="Final-demo prefix variant",
        yaxis_title="SAE feature",
        height=max(450, 26 * len(feature_ids) + 180),
        width=max(850, 80 * N_PREFIX_VARIANTS + 350),
    )
    fig.show()


for lab in range(NUM_LABELS):
    plot_label_specific_heatmap(lab, normalize_to_baseline=False)

# Optional normalized heatmaps. These are useful for visually identifying LFH-sensitive features.
for lab in range(NUM_LABELS):
    plot_label_specific_heatmap(lab, normalize_to_baseline=True)

# Retention plot for all selected label-specific features.
fig_retention = px.scatter(
    label_specific_stability_df,
    x="label_specific_rank",
    y=f"retention_rate_top_{RETENTION_TOP_K}",
    color="label_word",
    symbol="stable_by_rule",
    hover_data=[
        "feature_idx",
        "baseline_label_specific_pre_gap",
        "min_over_baseline",
        "worst_variant_rank",
        "best_variant_rank",
    ],
    title=f"Single-prefix label-specific rank versus multi-prefix retention in top-{RETENTION_TOP_K}",
    labels={
        "label_specific_rank": "Original single-prefix label-specific rank",
        f"retention_rate_top_{RETENTION_TOP_K}": f"Retention rate in top-{RETENTION_TOP_K}",
        "label_word": "Gold label",
    },
)
fig_retention.add_hline(y=STABLE_MIN_RETENTION, line_dash="dash")
fig_retention.update_layout(height=500, width=900)
fig_retention.show()

# Gap range plot: how much the label-specific score moves when only the final demo changes.
fig_range = px.scatter(
    label_specific_stability_df,
    x="baseline_label_specific_pre_gap",
    y="range_label_specific_pre_gap_across_prefixes",
    color="label_word",
    symbol="stable_by_rule",
    hover_data=["feature_idx", "label_specific_rank", "min_over_baseline", f"retention_rate_top_{RETENTION_TOP_K}"],
    title="Label-specific preactivation gap instability under final-demo variation",
    labels={
        "baseline_label_specific_pre_gap": "Baseline label-specific preactivation gap",
        "range_label_specific_pre_gap_across_prefixes": "Range of gap across prefix variants",
        "label_word": "Gold label",
    },
)
fig_range.update_layout(height=500, width=900)
fig_range.show()


{
  "model": "Gemma2-2B-IT",
  "target_layer": 12,
  "n_queries": 100,
  "n_prefix_variants": 8,
  "label_specific_top_n_per_label": 20,
  "retention_top_k": 20,
  "stable_min_retention": 0.7,
  "stable_min_baseline_ratio": 0.5,
  "query_analysis_positions": [
    -2,
    -1
  ],
  "stable_count": 67,
  "sensitive_count": 13
}
Final-demo label counts by variant index:


final_demo_label_word,Business,Sports,Technology,World
variant_idx,,,,
0,24,19,31,26
1,29,26,27,18
2,29,23,20,28
3,23,26,27,24
4,24,26,24,26
5,29,29,23,19
6,23,29,22,26
7,19,22,26,33


In [8]:

# ============================================================
# 7. Optional: earlier-demo control
# ============================================================
# This control varies an earlier demonstration while keeping the final demonstration fixed.
# If LFH is truly local, varying the final demonstration should generally produce more
# instability in query-side label-specific features than varying a more distant demonstration.


def build_earlier_demo_control_prompts(control_slot: int = EARLIER_DEMO_CONTROL_SLOT):
    if control_slot < 1 or control_slot >= N_SHOT:
        raise ValueError("control_slot should be an earlier demo slot, e.g. 3, not the final slot.")

    # Reuse the same settings, but build a fresh controlled prompt set.
    rng_control = random.Random(seed + 999)
    test_indices = sample_label_balanced_test_indices(N_QUERY_TOTAL, rng_control)
    prompts_control = []
    metadata_control = []

    for q_local_idx, test_idx in enumerate(test_indices):
        query_row = test_data_agnews[int(test_idx)]

        # Base N-shot prefix.
        base_label_seq = balanced_label_sequence(N_SHOT, rng=rng_control)
        base_demo_indices = sample_train_indices_for_labels(base_label_seq, rng=rng_control)

        # Variants for the earlier slot, balanced by label. The final demo remains fixed.
        replacement_label_seq = balanced_label_sequence(N_PREFIX_VARIANTS, rng=rng_control)
        replacement_indices = sample_train_indices_for_labels(
            replacement_label_seq,
            rng=rng_control,
            exclude=base_demo_indices,
        )

        for variant_idx, repl_idx in enumerate(replacement_indices):
            demo_indices = list(base_demo_indices)
            demo_indices[control_slot - 1] = int(repl_idx)
            demo_rows = [train_data_agnews[int(i)] for i in demo_indices]
            prompt, query_char_span = render_single_user_prompt(demo_rows, query_row)
            prompts_control.append(prompt)
            metadata_control.append({
                "query_local_idx": int(q_local_idx),
                "variant_idx": int(variant_idx),
                "query_test_idx": int(test_idx),
                "query_label": int(query_row["label"]),
                "query_label_word": label_to_words_agnews[int(query_row["label"])],
                "varied_demo_slot": int(control_slot),
                "replacement_demo_idx": int(repl_idx),
                "replacement_demo_label": int(train_data_agnews[int(repl_idx)]["label"]),
                "replacement_demo_label_word": label_to_words_agnews[int(train_data_agnews[int(repl_idx)]["label"])],
                "fixed_final_demo_idx": int(base_demo_indices[-1]),
                "fixed_final_demo_label": int(train_data_agnews[int(base_demo_indices[-1])]["label"]),
                "fixed_final_demo_label_word": label_to_words_agnews[int(train_data_agnews[int(base_demo_indices[-1])]["label"])],
                "query_char_start": int(query_char_span[0]),
                "query_char_end": int(query_char_span[1]),
            })

    for meta, prompt in zip(metadata_control, prompts_control):
        all_positions = token_positions_for_char_span(
            prompt=prompt,
            char_start=meta["query_char_start"],
            char_end=meta["query_char_end"],
        )
        meta["all_query_token_positions"] = [int(x) for x in all_positions]
        meta["selected_query_token_positions"] = [int(x) for x in select_query_positions(all_positions)]
        meta["num_query_tokens"] = int(len(all_positions))
        meta["num_selected_query_tokens"] = int(len(meta["selected_query_token_positions"]))

    return prompts_control, metadata_control


if RUN_EARLIER_DEMO_CONTROL:
    prompts_control, metadata_control = build_earlier_demo_control_prompts(EARLIER_DEMO_CONTROL_SLOT)
    print(f"Built {len(prompts_control)} earlier-demo-control prompts.")

    pre_ctrl, post_ctrl, freq_ctrl, metadata_control_loaded = collect_query_feature_stats_for_prompts(
        prompts=prompts_control,
        metadata=metadata_control,
    )

    meta_ctrl_df = pd.DataFrame(metadata_control_loaded)
    query_ctrl_meta_df = meta_ctrl_df.drop_duplicates("query_local_idx").sort_values("query_local_idx").reset_index(drop=True)
    query_ctrl_labels = query_ctrl_meta_df["query_label"].to_numpy(dtype=np.int64)

    pre_ctrl_qvf = pre_ctrl.reshape(N_QUERY_TOTAL, N_PREFIX_VARIANTS, n_features)
    ctrl_label_mean_pre, ctrl_other_mean_pre = label_vs_other_means(pre_ctrl_qvf, query_ctrl_labels)
    ctrl_label_specific_pre_gap = ctrl_label_mean_pre - ctrl_other_mean_pre

    comparison_rows = []
    for lab, feat in candidate_label_feature_pairs:
        final_scores = label_specific_pre_gap[lab, :, feat]
        ctrl_scores = ctrl_label_specific_pre_gap[lab, :, feat]
        final_range = float(final_scores.max() - final_scores.min())
        ctrl_range = float(ctrl_scores.max() - ctrl_scores.min())
        comparison_rows.append({
            "label": int(lab),
            "label_word": label_to_words_agnews[int(lab)],
            "feature_idx": int(feat),
            "final_demo_variant_range": final_range,
            "earlier_demo_variant_range": ctrl_range,
            "final_over_earlier_range_ratio": float(final_range / (ctrl_range + 1e-12)),
        })

    locality_control_df = pd.DataFrame(comparison_rows).sort_values(
        by="final_over_earlier_range_ratio",
        ascending=False,
    ).reset_index(drop=True)
    display(locality_control_df.head(80))

    fig_locality = px.scatter(
        locality_control_df,
        x="earlier_demo_variant_range",
        y="final_demo_variant_range",
        color="label_word",
        hover_data=["feature_idx", "final_over_earlier_range_ratio"],
        title=f"Locality control: final-demo variation versus Example {EARLIER_DEMO_CONTROL_SLOT} variation",
        labels={
            "earlier_demo_variant_range": f"Range when varying Example {EARLIER_DEMO_CONTROL_SLOT}",
            "final_demo_variant_range": "Range when varying final example",
            "label_word": "Gold label",
        },
    )
    max_lim = max(
        float(locality_control_df["earlier_demo_variant_range"].max()),
        float(locality_control_df["final_demo_variant_range"].max()),
    )
    fig_locality.add_shape(
        type="line",
        x0=0,
        y0=0,
        x1=max_lim,
        y1=max_lim,
        line=dict(dash="dash"),
    )
    fig_locality.update_layout(height=550, width=750)
    fig_locality.show()
else:
    print("RUN_EARLIER_DEMO_CONTROL is False. Set it to True in the config cell to run this optional control.")


RUN_EARLIER_DEMO_CONTROL is False. Set it to True in the config cell to run this optional control.



# 8. Causal validation: are multi-prefix-stable label features more causally relevant?

This section tests the hypothesis behind the LFH-aware stability filter:

- **Stable group:** label-specific features that remain label-specific across final-demo variants.
- **LFH-sensitive group:** label-specific features that looked strong under the original single-prefix prompt but failed the multi-prefix stability rule.

For every selected `(label, feature)` pair, we evaluate prompts whose gold query label is that label and intervene at the same query-token positions used for feature measurement.

Two interventions are run:

1. **Active-feature ablation:** when the feature is active at an edited token, subtract its current SAE reconstruction contribution:
   `hidden <- hidden - activation_f * W_dec[f]`.
2. **Feature amplification:** add the feature decoder direction with `alpha = 10`:
   `hidden <- hidden + 10 * W_dec[f]`.

The outcome is the model's next-token probability for the gold label, normalized over the four AGNews label tokens. The main quantities are:

- `ablation_decrease_p_label = baseline P(L) - ablated P(L)`; larger is stronger causal support for the label.
- `amplification_increase_p_label = amplified P(L) - baseline P(L)`; larger is stronger causal support for the label.


In [11]:

# ============================================================
# 8. Causal validation of stable versus LFH-sensitive features
# ============================================================

# Causal validation settings.
# By default this evaluates all 80 selected label-feature pairs: top-20 per AGNews label.
CAUSAL_MAX_PAIRS = None          # None = all selected pairs; set e.g. 8 for a quick smoke test
CAUSAL_BATCH_SIZE = BATCH_SIZE   # reduce if you hit memory limits
CAUSAL_AMPLIFICATION_ALPHA = 100.0
CAUSAL_BOOTSTRAP_SAMPLES = 2000

# The causal outcome is P(label) normalized over the four label words. This is usually the
# most interpretable probability for a classification prompt.
CAUSAL_PROBABILITY_MODE = "label_conditional"  # fixed in this cell

print("CAUSAL_MAX_PAIRS:", CAUSAL_MAX_PAIRS)
print("CAUSAL_BATCH_SIZE:", CAUSAL_BATCH_SIZE)
print("CAUSAL_AMPLIFICATION_ALPHA:", CAUSAL_AMPLIFICATION_ALPHA)
print("CAUSAL_PROBABILITY_MODE:", CAUSAL_PROBABILITY_MODE)


def get_single_token_label_ids():
    """Return one tokenizer id per AGNews label word, with sanity checks."""
    label_ids = []
    rows = []
    for lab in range(NUM_LABELS):
        word = label_to_words_agnews[lab]
        ids = tokenizer(word, add_special_tokens=False)["input_ids"]
        rows.append({"label": lab, "label_word": word, "token_ids": ids, "n_tokens": len(ids)})
        if len(ids) != 1:
            raise ValueError(
                f"Label word {word!r} tokenized into {len(ids)} tokens: {ids}. "
                "This causal cell expects single-token labels."
            )
        label_ids.append(int(ids[0]))
    display(pd.DataFrame(rows))
    return label_ids


LABEL_TOKEN_IDS = get_single_token_label_ids()
LABEL_TOKEN_IDS_TENSOR = torch.tensor(LABEL_TOKEN_IDS, dtype=torch.long, device=get_model_input_device(model))


def label_conditional_probs_from_logits(logits: torch.Tensor) -> torch.Tensor:
    """
    Return P(label) normalized over the four AGNews label tokens.

    logits: [B, S, vocab]. We use logits at the final prompt token to score the next token.
    Returns: [B, NUM_LABELS].
    """
    label_token_ids = LABEL_TOKEN_IDS_TENSOR.to(logits.device)
    next_logits = logits[:, -1, :].float()
    label_logits = next_logits[:, label_token_ids]
    return torch.softmax(label_logits, dim=-1)


@torch.no_grad()
def collect_baseline_label_probs(prompts, batch_size: int = CAUSAL_BATCH_SIZE):
    """Collect baseline next-token P(label), normalized across the four label tokens, for every prompt."""
    model_device = get_model_input_device(model)
    probs = np.zeros((len(prompts), NUM_LABELS), dtype=np.float32)
    model.eval()

    for start in range(0, len(prompts), batch_size):
        end = min(start + batch_size, len(prompts))
        enc = tokenizer(
            prompts[start:end],
            padding=True,
            return_tensors="pt",
            add_special_tokens=False,
        )
        input_ids = enc["input_ids"].to(model_device)
        attention_mask = enc["attention_mask"].to(model_device)

        out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        label_probs = label_conditional_probs_from_logits(out.logits)
        probs[start:end] = label_probs.detach().cpu().numpy()

        del out, input_ids, attention_mask, enc, label_probs
        print(f"Baseline probabilities: {end}/{len(prompts)}")

    return probs


baseline_label_probs_by_prompt = collect_baseline_label_probs(prompts)
print("baseline_label_probs_by_prompt:", baseline_label_probs_by_prompt.shape)


# Build the label-feature pairs to test. Each row corresponds to the same pair used in the
# stability table, so duplicated feature ids across labels are intentionally retained.
causal_pairs_df = label_specific_stability_df.copy().reset_index(drop=False).rename(columns={"index": "pair_id"})
if CAUSAL_MAX_PAIRS is not None:
    # Balanced quick mode: take from stable and sensitive groups separately when possible.
    stable_part = causal_pairs_df[causal_pairs_df["stable_by_rule"]].head(int(CAUSAL_MAX_PAIRS) // 2)
    sensitive_part = causal_pairs_df[~causal_pairs_df["stable_by_rule"]].head(int(CAUSAL_MAX_PAIRS) - len(stable_part))
    causal_pairs_df = pd.concat([stable_part, sensitive_part], ignore_index=True)

causal_pairs_df["causal_group"] = np.where(causal_pairs_df["stable_by_rule"], "multi-prefix stable", "LFH-sensitive")
print("Causal pairs to test:", len(causal_pairs_df))
display(causal_pairs_df[[
    "pair_id", "label_word", "feature_idx", "label_specific_rank", "stable_by_rule",
    "baseline_label_specific_pre_gap", f"retention_rate_top_{RETENTION_TOP_K}", "min_over_baseline", "worst_variant_rank"
]])


# Construct one intervention instance per (label-feature pair, prompt whose query label matches that label).
instance_rows = []
for _, pair in causal_pairs_df.iterrows():
    lab = int(pair["label"])
    feat = int(pair["feature_idx"])
    pair_id = int(pair["pair_id"])
    prompt_indices_for_label = meta_df.index[meta_df["query_label"].to_numpy(dtype=np.int64) == lab].to_numpy(dtype=np.int64)
    for prompt_idx in prompt_indices_for_label:
        meta = prompt_metadata[int(prompt_idx)]
        instance_rows.append({
            "pair_id": pair_id,
            "prompt_idx": int(prompt_idx),
            "query_local_idx": int(meta["query_local_idx"]),
            "variant_idx": int(meta["variant_idx"]),
            "label": lab,
            "label_word": label_to_words_agnews[lab],
            "feature_idx": feat,
        })

causal_instances_df = pd.DataFrame(instance_rows)
print("Causal intervention instances:", len(causal_instances_df))
display(causal_instances_df.head())


def _selected_padded_positions_for_batch(batch_meta, attention_mask, batch_seq_len: int):
    """Convert saved unpadded selected query positions into padded batch positions."""
    selected_positions_by_row = []
    for b, meta in enumerate(batch_meta):
        unpadded_len = int(attention_mask[b].sum().item())
        pad_shift = int(batch_seq_len - unpadded_len)
        positions = [pad_shift + int(pos) for pos in meta["selected_query_token_positions"]]
        positions = [pos for pos in positions if 0 <= pos < batch_seq_len]
        if not positions:
            raise RuntimeError("No valid selected token positions after padding adjustment.")
        selected_positions_by_row.append(positions)
    return selected_positions_by_row


@torch.no_grad()
def collect_intervened_gold_label_probs(instances_df: pd.DataFrame, mode: str, batch_size: int = CAUSAL_BATCH_SIZE):
    """
    Run feature-level interventions for prompt-feature instances.

    mode:
      - 'ablate_active': subtract current post-activation_f * W_dec[f] on selected query tokens.
      - 'amplify': add CAUSAL_AMPLIFICATION_ALPHA * W_dec[f] on selected query tokens.

    Returns: [n_instances] containing P(gold label) after intervention.
    """
    if mode not in {"ablate_active", "amplify"}:
        raise ValueError(f"Unknown intervention mode: {mode}")

    n = len(instances_df)
    gold_probs = np.zeros(n, dtype=np.float32)
    model_device = get_model_input_device(model)
    block = model.get_submodule(f"model.layers.{TARGET_LAYER}")
    model.eval()
    sae.eval()

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_rows = instances_df.iloc[start:end]
        batch_prompt_indices = batch_rows["prompt_idx"].to_numpy(dtype=np.int64)
        batch_prompts = [prompts[int(i)] for i in batch_prompt_indices]
        batch_meta = [prompt_metadata[int(i)] for i in batch_prompt_indices]
        batch_features = batch_rows["feature_idx"].to_numpy(dtype=np.int64)
        batch_labels = batch_rows["label"].to_numpy(dtype=np.int64)

        enc = tokenizer(
            batch_prompts,
            padding=True,
            return_tensors="pt",
            add_special_tokens=False,
        )
        input_ids = enc["input_ids"].to(model_device)
        attention_mask = enc["attention_mask"].to(model_device)
        batch_seq_len = int(input_ids.shape[1])
        selected_positions_by_row = _selected_padded_positions_for_batch(batch_meta, attention_mask, batch_seq_len)

        def hook_fn(module, inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            hidden_mod = hidden.clone()
            hidden_device = hidden_mod.device
            hidden_dtype = hidden_mod.dtype

            for b, feat in enumerate(batch_features):
                feat = int(feat)
                pos = selected_positions_by_row[b]
                x = hidden[b, pos, :]
                decoder_dir = sae.W_dec[feat].to(device=hidden_device, dtype=hidden_dtype)

                if mode == "ablate_active":
                    # Compute the current SAE post-activation and remove only the active contribution.
                    pre = sae.encode_pre(x).float()
                    post = sae.activation_fn(pre.to(dtype=sae.dtype)).float()
                    coeff = post[:, feat].to(device=hidden_device, dtype=hidden_dtype)
                    hidden_mod[b, pos, :] = hidden_mod[b, pos, :] - coeff[:, None] * decoder_dir[None, :]
                elif mode == "amplify":
                    hidden_mod[b, pos, :] = hidden_mod[b, pos, :] + float(CAUSAL_AMPLIFICATION_ALPHA) * decoder_dir[None, :]

            if isinstance(output, tuple):
                return (hidden_mod,) + output[1:]
            return hidden_mod

        handle = block.register_forward_hook(hook_fn)
        try:
            out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        finally:
            handle.remove()

        label_probs = label_conditional_probs_from_logits(out.logits).detach().cpu().numpy()
        gold_probs[start:end] = label_probs[np.arange(end - start), batch_labels]

        del out, input_ids, attention_mask, enc, label_probs
        print(f"{mode}: {end}/{n}")

    return gold_probs


# Baseline summaries per pair using the same prompt subset as each intervention.
baseline_rows = []
for _, pair in causal_pairs_df.iterrows():
    pair_id = int(pair["pair_id"])
    lab = int(pair["label"])
    feat = int(pair["feature_idx"])
    prompt_indices = causal_instances_df.loc[causal_instances_df["pair_id"] == pair_id, "prompt_idx"].to_numpy(dtype=np.int64)

    baseline_gold = baseline_label_probs_by_prompt[prompt_indices, lab]
    feature_freq = freq_by_prompt[prompt_indices, feat]
    feature_post = post_by_prompt[prompt_indices, feat]
    feature_pre = pre_by_prompt[prompt_indices, feat]

    baseline_rows.append({
        "pair_id": pair_id,
        "baseline_mean_p_label": float(np.mean(baseline_gold)),
        "baseline_median_p_label": float(np.median(baseline_gold)),
        "n_prompts_for_label": int(len(prompt_indices)),
        "feature_active_prompt_fraction": float(np.mean(feature_freq > 0)),
        "mean_feature_preactivation_on_gold_label_prompts": float(np.mean(feature_pre)),
        "mean_feature_postactivation_on_gold_label_prompts": float(np.mean(feature_post)),
    })

baseline_pair_df = pd.DataFrame(baseline_rows)

# Run the two causal interventions.
ablation_gold_probs = collect_intervened_gold_label_probs(causal_instances_df, mode="ablate_active")
amplified_gold_probs = collect_intervened_gold_label_probs(causal_instances_df, mode="amplify")

causal_instances_df = causal_instances_df.copy()
causal_instances_df["baseline_p_label"] = baseline_label_probs_by_prompt[
    causal_instances_df["prompt_idx"].to_numpy(dtype=np.int64),
    causal_instances_df["label"].to_numpy(dtype=np.int64),
]
causal_instances_df["ablated_p_label"] = ablation_gold_probs
causal_instances_df["amplified_p_label"] = amplified_gold_probs
causal_instances_df["ablation_decrease_p_label"] = causal_instances_df["baseline_p_label"] - causal_instances_df["ablated_p_label"]
causal_instances_df["amplification_increase_p_label"] = causal_instances_df["amplified_p_label"] - causal_instances_df["baseline_p_label"]

# Aggregate intervention effects per label-feature pair.
causal_pair_effects_df = (
    causal_instances_df
    .groupby("pair_id")
    .agg(
        ablated_mean_p_label=("ablated_p_label", "mean"),
        amplified_mean_p_label=("amplified_p_label", "mean"),
        mean_ablation_decrease_p_label=("ablation_decrease_p_label", "mean"),
        median_ablation_decrease_p_label=("ablation_decrease_p_label", "median"),
        mean_amplification_increase_p_label=("amplification_increase_p_label", "mean"),
        median_amplification_increase_p_label=("amplification_increase_p_label", "median"),
        frac_prompts_ablation_decreases_p_label=("ablation_decrease_p_label", lambda x: float(np.mean(np.asarray(x) > 0))),
        frac_prompts_amplification_increases_p_label=("amplification_increase_p_label", lambda x: float(np.mean(np.asarray(x) > 0))),
    )
    .reset_index()
)

causal_validation_df = (
    causal_pairs_df
    .merge(baseline_pair_df, on="pair_id", how="left")
    .merge(causal_pair_effects_df, on="pair_id", how="left")
)

# Convenience columns for reading.
causal_validation_df["ablation_decrease_pp"] = 100.0 * causal_validation_df["mean_ablation_decrease_p_label"]
causal_validation_df["amplification_increase_pp"] = 100.0 * causal_validation_df["mean_amplification_increase_p_label"]
causal_validation_df["baseline_p_label_pct"] = 100.0 * causal_validation_df["baseline_mean_p_label"]
causal_validation_df["ablated_p_label_pct"] = 100.0 * causal_validation_df["ablated_mean_p_label"]
causal_validation_df["amplified_p_label_pct"] = 100.0 * causal_validation_df["amplified_mean_p_label"]

print("Causal validation per label-feature pair:")
display(causal_validation_df[[
    "label_word", "feature_idx", "label_specific_rank", "stable_by_rule", "causal_group",
    "baseline_label_specific_pre_gap", f"retention_rate_top_{RETENTION_TOP_K}", "min_over_baseline",
    "feature_active_prompt_fraction", "baseline_p_label_pct", "ablated_p_label_pct", "amplified_p_label_pct",
    "ablation_decrease_pp", "amplification_increase_pp",
    "frac_prompts_ablation_decreases_p_label", "frac_prompts_amplification_increases_p_label",
]].sort_values(["stable_by_rule", "label_word", "label_specific_rank"], ascending=[False, True, True]))


def bootstrap_mean_ci(values, n_boot: int = CAUSAL_BOOTSTRAP_SAMPLES, seed: int = seed, ci: float = 0.95):
    vals = np.asarray(values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan, np.nan, np.nan
    mean = float(vals.mean())
    if vals.size == 1 or n_boot <= 0:
        return mean, np.nan, np.nan
    rng_boot = np.random.default_rng(seed)
    draws = rng_boot.choice(vals, size=(int(n_boot), vals.size), replace=True).mean(axis=1)
    lo = float(np.quantile(draws, (1.0 - ci) / 2.0))
    hi = float(np.quantile(draws, 1.0 - (1.0 - ci) / 2.0))
    return mean, lo, hi


group_rows = []
for stable_value, group_name in [(True, "multi-prefix stable"), (False, "LFH-sensitive")]:
    g = causal_validation_df[causal_validation_df["stable_by_rule"] == stable_value]
    row = {
        "causal_group": group_name,
        "n_label_feature_pairs": int(len(g)),
        "mean_active_prompt_fraction": float(g["feature_active_prompt_fraction"].mean()) if len(g) else np.nan,
        "median_active_prompt_fraction": float(g["feature_active_prompt_fraction"].median()) if len(g) else np.nan,
    }
    for metric, label in [
        ("mean_ablation_decrease_p_label", "ablation_decrease"),
        ("mean_amplification_increase_p_label", "amplification_increase"),
    ]:
        mean, lo, hi = bootstrap_mean_ci(g[metric].to_numpy(), seed=seed + (1 if stable_value else 2))
        row[f"mean_{label}"] = mean
        row[f"mean_{label}_ci_low"] = lo
        row[f"mean_{label}_ci_high"] = hi
        row[f"mean_{label}_pp"] = 100.0 * mean if np.isfinite(mean) else np.nan
        row[f"mean_{label}_ci_low_pp"] = 100.0 * lo if np.isfinite(lo) else np.nan
        row[f"mean_{label}_ci_high_pp"] = 100.0 * hi if np.isfinite(hi) else np.nan
    group_rows.append(row)

causal_group_summary_df = pd.DataFrame(group_rows)
print("Stable versus LFH-sensitive causal summary:")
display(causal_group_summary_df)

# Label-level group summary, useful for finding whether one label dominates the comparison.
label_group_summary_df = (
    causal_validation_df
    .groupby(["label_word", "causal_group"])
    .agg(
        n_label_feature_pairs=("feature_idx", "count"),
        mean_ablation_decrease_pp=("ablation_decrease_pp", "mean"),
        mean_amplification_increase_pp=("amplification_increase_pp", "mean"),
        mean_active_prompt_fraction=("feature_active_prompt_fraction", "mean"),
    )
    .reset_index()
)
print("Per-label stable/sensitive causal summary:")
display(label_group_summary_df)

# Plotly comparison: ablation and amplification effects by stability group.
plot_df = causal_validation_df.copy()
plot_df["stability_group"] = np.where(plot_df["stable_by_rule"], "multi-prefix stable", "LFH-sensitive")

fig_ablation = px.box(
    plot_df,
    x="stability_group",
    y="ablation_decrease_pp",
    color="label_word",
    points="all",
    hover_data=["feature_idx", "label_specific_rank", "feature_active_prompt_fraction", f"retention_rate_top_{RETENTION_TOP_K}", "min_over_baseline"],
    title="Feature ablation: decrease in gold-label probability by multi-prefix stability group",
    labels={
        "stability_group": "Feature group",
        "ablation_decrease_pp": "Baseline P(L) - ablated P(L), percentage points",
        "label_word": "Gold label",
    },
)
fig_ablation.add_hline(y=0.0, line_dash="dash")
fig_ablation.update_layout(height=550, width=950)
fig_ablation.show()

fig_amplification = px.box(
    plot_df,
    x="stability_group",
    y="amplification_increase_pp",
    color="label_word",
    points="all",
    hover_data=["feature_idx", "label_specific_rank", "feature_active_prompt_fraction", f"retention_rate_top_{RETENTION_TOP_K}", "min_over_baseline"],
    title=f"Feature amplification (alpha={CAUSAL_AMPLIFICATION_ALPHA:g}): increase in gold-label probability by stability group",
    labels={
        "stability_group": "Feature group",
        "amplification_increase_pp": "Amplified P(L) - baseline P(L), percentage points",
        "label_word": "Gold label",
    },
)
fig_amplification.add_hline(y=0.0, line_dash="dash")
fig_amplification.update_layout(height=550, width=950)
fig_amplification.show()

# Scatter: stability score versus causal effect. Ideally, more stable features have stronger causal effects.
fig_scatter = px.scatter(
    plot_df,
    x=f"retention_rate_top_{RETENTION_TOP_K}",
    y="ablation_decrease_pp",
    color="label_word",
    symbol="stable_by_rule",
    size="feature_active_prompt_fraction",
    hover_data=["feature_idx", "label_specific_rank", "amplification_increase_pp", "min_over_baseline", "worst_variant_rank"],
    title="Does multi-prefix retention predict feature-ablation causal relevance?",
    labels={
        f"retention_rate_top_{RETENTION_TOP_K}": f"Retention rate in top-{RETENTION_TOP_K}",
        "ablation_decrease_pp": "Ablation decrease in P(L), percentage points",
        "label_word": "Gold label",
    },
)
fig_scatter.add_hline(y=0.0, line_dash="dash")
fig_scatter.add_vline(x=STABLE_MIN_RETENTION, line_dash="dash")
fig_scatter.update_layout(height=550, width=950)
fig_scatter.show()

# Compact interpretation helper.
stable_mean_ablate = causal_group_summary_df.loc[causal_group_summary_df["causal_group"] == "multi-prefix stable", "mean_ablation_decrease_pp"].iloc[0]
sens_mean_ablate = causal_group_summary_df.loc[causal_group_summary_df["causal_group"] == "LFH-sensitive", "mean_ablation_decrease_pp"].iloc[0]
stable_mean_amp = causal_group_summary_df.loc[causal_group_summary_df["causal_group"] == "multi-prefix stable", "mean_amplification_increase_pp"].iloc[0]
sens_mean_amp = causal_group_summary_df.loc[causal_group_summary_df["causal_group"] == "LFH-sensitive", "mean_amplification_increase_pp"].iloc[0]

print("=" * 100)
print("Quick readout")
print("=" * 100)
print(f"Mean ablation decrease, stable group:      {stable_mean_ablate:.4f} percentage points")
print(f"Mean ablation decrease, LFH-sensitive:     {sens_mean_ablate:.4f} percentage points")
print(f"Mean amplification increase, stable group: {stable_mean_amp:.4f} percentage points")
print(f"Mean amplification increase, sensitive:    {sens_mean_amp:.4f} percentage points")
print()
if stable_mean_ablate > sens_mean_ablate and stable_mean_amp > sens_mean_amp:
    print("Pattern matches the ideal validation: multi-prefix-stable features are more causally relevant on both tests.")
elif stable_mean_ablate > sens_mean_ablate or stable_mean_amp > sens_mean_amp:
    print("Pattern is partially supportive: stable features are stronger on one causal test but not both.")
else:
    print("Pattern is not supportive under this run/settings; inspect label-wise summaries and active fractions.")


CAUSAL_MAX_PAIRS: None
CAUSAL_BATCH_SIZE: 4
CAUSAL_AMPLIFICATION_ALPHA: 100.0
CAUSAL_PROBABILITY_MODE: label_conditional


,label,label_word,token_ids,n_tokens
0,0,World,[10772],1
1,1,Sports,[27093],1
2,2,Business,[14103],1
3,3,Technology,[39364],1


Baseline probabilities: 4/800
Baseline probabilities: 8/800
Baseline probabilities: 12/800
Baseline probabilities: 16/800
Baseline probabilities: 20/800
Baseline probabilities: 24/800
Baseline probabilities: 28/800
Baseline probabilities: 32/800
Baseline probabilities: 36/800
Baseline probabilities: 40/800
Baseline probabilities: 44/800
Baseline probabilities: 48/800
Baseline probabilities: 52/800
Baseline probabilities: 56/800
Baseline probabilities: 60/800
Baseline probabilities: 64/800
Baseline probabilities: 68/800
Baseline probabilities: 72/800
Baseline probabilities: 76/800
Baseline probabilities: 80/800
Baseline probabilities: 84/800
Baseline probabilities: 88/800
Baseline probabilities: 92/800
Baseline probabilities: 96/800
Baseline probabilities: 100/800
Baseline probabilities: 104/800
Baseline probabilities: 108/800
Baseline probabilities: 112/800
Baseline probabilities: 116/800
Baseline probabilities: 120/800
Baseline probabilities: 124/800
Baseline probabilities: 128/800
Ba

,pair_id,label_word,feature_idx,label_specific_rank,stable_by_rule,baseline_label_specific_pre_gap,retention_rate_top_20,min_over_baseline,worst_variant_rank
0,0,World,14090,1,True,6.002006,1.000,0.999478,1
1,1,World,1041,2,True,4.656667,1.000,0.516106,10
2,2,World,5185,3,True,3.586107,1.000,0.981599,3
3,3,World,14599,4,True,3.075547,1.000,0.936918,5
4,4,World,2600,5,True,2.712956,1.000,0.838079,8
...,...,...,...,...,...,...,...,...,...
75,75,Technology,11807,16,False,1.578333,0.375,0.586326,81
76,76,Technology,11970,17,True,1.436165,0.875,0.827065,22
77,77,Technology,9059,18,False,1.354323,0.125,0.284400,1525
78,78,Technology,4626,19,False,1.353281,0.125,0.500717,215


Causal intervention instances: 16000


...

Stable versus LFH-sensitive causal summary:


,causal_group,n_label_feature_pairs,mean_active_prompt_fraction,median_active_prompt_fraction,mean_ablation_decrease,mean_ablation_decrease_ci_low,mean_ablation_decrease_ci_high,mean_ablation_decrease_pp,mean_ablation_decrease_ci_low_pp,mean_ablation_decrease_ci_high_pp,mean_amplification_increase,mean_amplification_increase_ci_low,mean_amplification_increase_ci_high,mean_amplification_increase_pp,mean_amplification_increase_ci_low_pp,mean_amplification_increase_ci_high_pp
0,multi-prefix stable,67,0.473358,0.485,0.000462,0.000234,0.000693,0.046170,0.023365,0.069340,-0.010664,-0.022420,0.001685,-1.066442,-2.241962,0.168500
1,LFH-sensitive,13,0.220769,0.195,0.000108,-0.000017,0.000273,0.010785,-0.001665,0.027328,-0.005370,-0.022155,0.010884,-0.536964,-2.215477,1.088352


Per-label stable/sensitive causal summary:


,label_word,causal_group,n_label_feature_pairs,mean_ablation_decrease_pp,mean_amplification_increase_pp,mean_active_prompt_fraction
0,Business,LFH-sensitive,2,0.001086,-2.019988,0.387500
1,Business,multi-prefix stable,18,0.048498,-5.102559,0.451389
2,Sports,LFH-sensitive,4,0.002151,-3.719553,0.210000
3,Sports,multi-prefix stable,16,0.066264,-4.129187,0.580000
4,Technology,LFH-sensitive,5,0.004741,1.919064,0.088000
5,Technology,multi-prefix stable,15,-0.003553,4.459754,0.355000
6,World,LFH-sensitive,2,0.052865,1.171169,0.407500
7,World,multi-prefix stable,18,0.067417,1.086952,0.499167


Quick readout
Mean ablation decrease, stable group:      0.0462 percentage points
Mean ablation decrease, LFH-sensitive:     0.0108 percentage points
Mean amplification increase, stable group: -1.0664 percentage points
Mean amplification increase, sensitive:    -0.5370 percentage points

Pattern is partially supportive: stable features are stronger on one causal test but not both.


# [-1, -2], alpha = 50

In [ ]:
====================================================================================================
Quick readout
====================================================================================================
Mean ablation decrease, stable group:      0.0462 percentage points
Mean ablation decrease, LFH-sensitive:     0.0108 percentage points
Mean amplification increase, stable group: 0.6023 percentage points
Mean amplification increase, sensitive:    0.5040 percentage points

Pattern matches the ideal validation: multi-prefix-stable features are more causally relevant on both tests.

---

# [-2]

# alpha = 50

In [ ]:
====================================================================================================
Quick readout
====================================================================================================
Mean ablation decrease, stable group:      0.0406 percentage points
Mean ablation decrease, LFH-sensitive:     -0.0024 percentage points
Mean amplification increase, stable group: 1.0648 percentage points
Mean amplification increase, sensitive:    0.4336 percentage points

Pattern matches the ideal validation: multi-prefix-stable features are more causally relevant on both tests.

# alpha = 10

In [ ]:
====================================================================================================
Quick readout
====================================================================================================
Mean ablation decrease, stable group:      0.0406 percentage points
Mean ablation decrease, LFH-sensitive:     -0.0024 percentage points
Mean amplification increase, stable group: 0.2099 percentage points
Mean amplification increase, sensitive:    0.0994 percentage points

Pattern matches the ideal validation: multi-prefix-stable features are more causally relevant on both tests.


## How to report the causal validation

The key table to include in the rebuttal is `causal_group_summary_df`:

- `mean_ablation_decrease_pp`: how much gold-label probability falls when the selected feature is removed when active.
- `mean_amplification_increase_pp`: how much gold-label probability rises when the selected feature direction is added with `alpha=10`.

The intended validation pattern is:

```text
multi-prefix stable features  >  LFH-sensitive features
```

for both ablation decrease and amplification increase. If the pattern holds, the rebuttal can state that the multi-prefix stability filter is not merely a descriptive diagnostic: it preferentially retains features with stronger causal support for the intended label, while filtering out single-prefix features that are more likely to be local-history-sensitive artifacts.



## How to report this experiment

A compact rebuttal/table can report:

- that features are selected by **label-specific preactivation gap**, not by overall activation magnitude;
- for each AGNews label, the top-20 single-prefix label-specific features;
- the fraction of those features that pass the multi-prefix stability rule;
- examples of high-ranked label-specific features that are unstable when only the final demonstration changes;
- Plotly heatmaps showing label-specific preactivation gaps across final-demo variants.

Suggested interpretation:

> We add an LFH-aware multi-prefix stability check for label-specific SAE features. For each fixed query, only the final demonstration is resampled. A feature selected from the original prefix is retained as robust only if its label-specific preactivation gap remains high across these alternative prefixes. Features whose apparent label-specificity collapses or moves sharply under final-demo variation are flagged as LFH-sensitive rather than directly interpreted as query-semantic features.

This is intended as a lightweight diagnostic, not a complete causal decomposition.
